# Importing Libraries


In [ ]:
# Import Polars for dataframe operations
import polars as pl

# Import Path to manage fiel paths
from pathlib import Path

## Define Data Path


In [ ]:
# Define path to the data folder
data_folder = Path("../data")

# Load Cleaned Dataset


In [ ]:
# Load cleaned sales dataset
sales_df = pl.read_parquet(data_folder / "sales_cleaned.parquet")

# Preview Dataset
sales_df.head()

## Extract Purchase Month

skipped purchase_date to datetime since it already exists.


In [ ]:
# Extract purchase month
sales_df = sales_df.with_columns(
    pl.col("purchase_date")
    .dt.truncate("1mo")
    .alias("purchase_month")
)

# Quick Check
sales_df.select(["purchase_date", "purchase_month"]).head()

# Identify Customer First Purchase (Cohort)

Each customer belongs to the month they first purchased.


In [ ]:
# Find first purchase month for each customer
cohort_df = sales_df.group_by("customer_id").agg(
    pl.col("purchase_month").min().alias("cohort_month")
)

cohort_df.head()

## Join Cohort Back to Transactions


In [ ]:
# Merge cohort info into sales data
sales_cohort_df = sales_df.join(cohort_df, on="customer_id")

# Quick Check
sales_cohort_df.head()

## Calculating Cohort Index

cohort index = months since first purchase


In [ ]:
# Calculate months since first purchase
sales_cohort_df = sales_cohort_df.with_columns(
    (
        (pl.col("purchase_month").dt.year() -
         pl.col("cohort_month").dt.year()) * 12
        + (pl.col("purchase_month").dt.month() -
           pl.col("cohort_month").dt.month())
    ).alias("cohort_index")
)

sales_cohort_df.head()

In [ ]:
# Quick Check
sales_cohort_df.select(
    ["purchase_month", "cohort_month", "cohort_index"]).tail(10)

## Count Active Customers Per Cohort


In [ ]:
# Count customers in each cohort period
cohort_counts = sales_cohort_df.group_by(
    ["cohort_month", "cohort_index"]
).agg(pl.col("customer_id").n_unique().alias("customers"))

cohort_counts.tail()

## Get Cohort Sizes


In [ ]:
# Get initial cohort size
cohort_sizes = cohort_counts.filter(pl.col("cohort_index") == 0
                                    ).select(["cohort_month", "customers"]
                                             ).rename({"customers": "cohort_size"})

## Calculate Retention Rate


In [ ]:
# Merge cohort size
cohort_retention = cohort_counts.join(
    cohort_sizes, on="cohort_month"
)

# Calculate retention rate
cohort_retention = cohort_retention.with_columns(
    (pl.col("customers") / pl.col("cohort_size")).alias("retention_rate")
)

cohort_retention.head()

## Save Dataset


In [ ]:
# Save cohort retention dataset
cohort_retention.write_csv(data_folder / "cohort_retention.csv")

# Save parquet version
cohort_retention.write_parquet(data_folder / "cohort_retention.parquet")